# Stage 6 · L2any → L2both merge / consolidation

Score candidate subtypes for cross-modality consistency (ROC-AUC / average precision per modality) and merge into the final **L2both subtypes** (206 total). Writes `merge{mcg,hic}_rocpr.npy`, consumed by the `merge_both` step. Example: **Epi-Ent**.

These are the original pipeline notebooks, concatenated in execution order with paths normalized to `ENTEX_ROOT`. They document the method and run per tissue / major type across the full raw dataset (heavy compute); they are shown for reference and are **not re-executed in the book**. Example group shown where templated.

In [1]:
# === Reproduction setup ===
import os, sys
ENTEX_ROOT = os.environ.get('ENTEX_ROOT', '/large_storage/zhoulab/zhoujt/project/ENTEx')
REF_ROOT   = os.environ.get('REF_ROOT',   '/large_storage/zhoulab/ref')
BOOK_ROOT  = os.environ.get('BOOK_ROOT',  f'{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas')
sys.path.insert(0, BOOK_ROOT); import repro_guard


## 6 · per-modality subtype merge scoring → rocpr

_Source: `clustering/merged/L2_hiconly/Epi-Ent/01.embed_final.ipynb`_

In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

import anndata
import scanpy as sc
import scanpy.external as sce
from sklearn.preprocessing import normalize
from sklearn.decomposition import TruncatedSVD

from ALLCools.clustering import *
from ALLCools.plot import *
from ALLCools.integration.seurat_class import SeuratIntegration

mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'

import warnings
warnings.filterwarnings("ignore")


In [3]:
def dump_embedding(adata, name, n_dim=2):
    # put manifold coordinates into adata.obs
    for i in range(n_dim):
        adata.obs[f'{name}_{i}'] = adata.obsm[f'X_{name}'][:, i]
    return adata


In [4]:
group_name = 'Neu-Exc'

In [5]:
# Parameters
cpu = 8
group_name = "Epi-Ent"
mem_gb = 32


In [6]:
adata_mc = anndata.read_h5ad('5kCG_embed.h5ad')
adata_all = anndata.read_h5ad('5kCG100k3C_embed.h5ad')
adata = anndata.read_h5ad('100k3C_embed.h5ad')
adata = adata[adata_all.obs.index].copy()


In [7]:
# npc_cg = [int(xx.split('_')[1][1:]) for xx in adata_mc.obsm.keys() if '_seuratL2_tsne' in xx][0]
# # npc_3c = [int(xx.split('_')[1][1:]) for xx in adata_3c.obsm.keys() if '100k3C_u' in xx][0]
# npc_3c = [int(xx.split('_')[1][2:]) for xx in adata_3c.obsm.keys() if '_seuratL2_tsne' in xx][0]
npc_cg, npc_3c = pd.read_csv('/large_storage/zhoulab/zhoujt/project/ENTEx/clustering/merged/npc.tsv', sep='\t', header=None, index_col=0).loc[group_name].values
print(npc_cg, npc_3c)


In [8]:
# clustering name
clustering_name = 'L1'

# ConsensusClustering
# Important factores
n_neighbors = 25
leiden_resolution = 1.0
# this parameter is the final target that limit the total number of clusters
# Higher accuracy means more conservative clustering results and less number of clusters
target_accuracy = 0.9
min_cluster_size = 30

# Other ConsensusClustering parameters
metric = 'euclidean'
consensus_rate = 0.8
leiden_repeats = 500
random_state = 0
train_frac = 0.5
train_max_n = 500
max_iter = 50
n_jobs = 8

# Dendrogram via Multiscale Bootstrap Resampling
nboot = 10000
method_dist = 'correlation'
method_hclust = 'average'

plot_type = 'static'

## hic clustering

In [9]:
ds = 2
coord_base = 'tsne'
adata.obsm[f'X_{coord_base}'] = adata.obsm[f'100k3C_pc{npc_3c}_seuratL2_{coord_base}'].copy()
dump_embedding(adata, coord_base)

fig, axes = plt.subplots(2, 3, figsize=(15, 8), dpi=300, constrained_layout=True)

tmp = adata.obs.copy()
ax = axes[0, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Donor',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        # palette='tab20',
                        scatter_kws={'rasterized':True},
                        show_legend=True
                       )

ax = axes[0, 1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Tissue',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        show_legend=True
                       )

# ax = axes[0, 2]
# _ = categorical_scatter(data=tmp,
#                         ax=ax,
#                         coord_base=coord_base,
#                         hue='leiden_cons',
#                         text_anno='leiden_cons', 
#                         s=ds,
#                         labelsize=8,
#                         max_points=None,
#                         palette='tab20',
#                         scatter_kws={'rasterized':True},
#                         # legend_kws={'ncol':1}, 
#                         # show_legend=True
#                        )

ax = axes[0, 2]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='ClusterTissue',
                        text_anno='ClusterTissue', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1}, 
                        # show_legend=True
                       )


ax = axes[1, 0]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue=np.log10(tmp['FinalmCReads']+1),
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )

ax = axes[1, 1]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue='mCGFrac',
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )

ax = axes[1, 2]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue='mCHFrac',
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )



In [10]:
cc = ConsensusClustering(model=None,
                         n_neighbors=n_neighbors,
                         metric=metric,
                         min_cluster_size=min_cluster_size,
                         leiden_repeats=leiden_repeats,
                         leiden_resolution=leiden_resolution,
                         consensus_rate=consensus_rate,
                         random_state=random_state,
                         train_frac=train_frac,
                         train_max_n=train_max_n,
                         max_iter=max_iter,
                         n_jobs=n_jobs,
                         target_accuracy=target_accuracy)


In [11]:
cc.fit_predict(adata.obsm[f'100k3C_pc{npc_3c}_seuratL2'])

In [12]:
adata.obs['leiden_cons'] = cc.label.copy()
adata.obs['leiden_cv'] = cc.cv_predicted_label.copy()


In [13]:
if len(cc.step_data)>1:
    cc.target_accuracy = 0
    cc.supervise_learning()
    cc.final_evaluation()
    adata.obs['leiden_init'] = cc.label.copy()
else:
    adata.obs['leiden_init'] = adata.obs['leiden_cons'].copy()
    

In [14]:
cc.save('ConcensusClustering_3c.model.lib')
adata.write_h5ad('100k3C_embed.h5ad')


In [15]:
from sklearn.metrics import adjusted_rand_score as ARI
ARI(adata.obs['leiden_cons'], adata.obs['leiden_cv'])


In [16]:
ds = 2
coord_base = 'tsne'
adata.obsm[f'X_{coord_base}'] = adata.obsm[f'100k3C_pc{npc_3c}_seuratL2_{coord_base}'].copy()
dump_embedding(adata, coord_base)

fig, axes = plt.subplots(2, 2, figsize=(10, 8), dpi=300, constrained_layout=True)

tmp = adata.obs.copy()
ax = axes[0,0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='leiden_cons',
                        text_anno='leiden_cons',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # show_legend=True
                       )

ax = axes[0,1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='leiden_cv',
                        text_anno='leiden_cv',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # show_legend=True
                       )

ax = axes[1,0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Tissue',
                        # text_anno='celltype', 
                        s=ds,
                        labelsize=6,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True
                       )

ax = axes[1,1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='ClusterTissue',
                        text_anno='ClusterTissue', 
                        s=ds,
                        labelsize=6,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1}, 
                        # show_legend=True
                       )

# plt.savefig(f'{group_name}_5kCG100k3C_cluster.pdf', transparent=True)


In [17]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score


In [18]:
data = normalize(adata.obsm[f'100k3C_pc{npc_3c}_seuratL2'], axis=1)
clf = LogisticRegression(class_weight='balanced')
skf = StratifiedKFold(n_splits=3, shuffle=True)
label = adata.obs['leiden_init'].copy()
leg = np.sort(label.unique())
while len(leg)>1:
    count = label.astype(str).value_counts()
    score = np.zeros((2, len(leg), len(leg)))
    for i in np.arange(len(leg)-1):
        for j in np.arange(i+1, len(leg)):
            selc = label.isin([leg[i], leg[j]])
            if count.loc[leg[i]]<count.loc[leg[j]]:
                label_dict = {leg[i]:1, leg[j]:0}
            else:
                label_dict = {leg[i]:0, leg[j]:1}
            # selc = np.random.permutation(np.where(selc)[0])
            X = data[selc]
            y = label.values[selc].map(label_dict)
            pred = np.zeros(len(y))
            for train_index, test_index in skf.split(X, y):
                X_train, X_test, y_train, y_test = X[train_index], X[test_index], y[train_index], y[test_index]
                pred[test_index] = clf.fit(X_train, y_train).predict_proba(X_test)[:,1]
            score[0,i,j] = roc_auc_score(y, pred)
            score[1,i,j] = average_precision_score(y, pred)
    score = np.min(score, axis=0)
    min_score = np.min(score[np.triu_indices(len(leg), k=1)])
    if min_score>0.9:
        break
    else:
        xx,yy = np.where(score==min_score)
        xx, yy = leg[xx[0]], leg[yy[0]]
        print(f'Merge {xx} {yy}')
        label[label==xx] = yy
        leg = np.sort(label.unique())

count = label.astype(str).value_counts()
labelmap = count.reset_index().reset_index().set_index('leiden_init')['index'].to_dict()
label_reorder = label.map(labelmap).astype(int)
label_reorder = 'c' + label_reorder.astype(str)
np.save('mergehic_rocpr.npy', label_reorder)


In [19]:
ds = 2
coord_base = 'tsne'
label_reorder = np.load('mergehic_rocpr.npy', allow_pickle=True)

fig, axes = plt.subplots(3, 3, figsize=(15, 12), dpi=300, constrained_layout=True)

for i,xx in enumerate([adata_all, adata_mc, adata]):
    dump_embedding(xx, coord_base)
    tmp = xx.obs.copy()
    tmp['leiden_init'] = label_reorder.copy()
    tmp['ClusterTissue'] = adata.obs['ClusterTissue'].copy()
    ax = axes[0, i]
    _ = categorical_scatter(data=tmp,
                            ax=ax,
                            coord_base=coord_base,
                            hue='Donor',
                            s=ds,
                            labelsize=8,
                            max_points=None,
                            # palette='tab20',
                            scatter_kws={'rasterized':True},
                            show_legend=True)

    ax = axes[1, i]
    _ = categorical_scatter(data=tmp,
                            ax=ax,
                            coord_base=coord_base,
                            hue='leiden_init',
                            text_anno='leiden_init', 
                            s=ds,
                            labelsize=8,
                            max_points=None,
                            palette='tab20',
                            scatter_kws={'rasterized':True},
                            legend_kws={'ncol':1}, 
                            show_legend=True)

    ax = axes[2, i]
    _ = categorical_scatter(data=tmp,
                            ax=ax,
                            coord_base=coord_base,
                            hue='ClusterTissue',
                            text_anno='ClusterTissue', 
                            s=ds,
                            labelsize=8,
                            max_points=None,
                            palette='tab20',
                            scatter_kws={'rasterized':True},
                            # legend_kws={'ncol':1}, 
                            # show_legend=False
                           )


## mc clustering

In [20]:
adata_3c = adata.copy()
adata = adata_mc.copy()


In [21]:
ds = 2
coord_base = 'tsne'
adata.obsm[f'X_{coord_base}'] = adata.obsm[f'5kCG_u{npc_cg}_seuratL2_{coord_base}'].copy()
dump_embedding(adata, coord_base)

fig, axes = plt.subplots(2, 3, figsize=(15, 8), dpi=300, constrained_layout=True)

tmp = adata.obs.copy()
ax = axes[0, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Donor',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        # palette='tab20',
                        scatter_kws={'rasterized':True},
                        show_legend=True
                       )

ax = axes[0, 1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Tissue',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        show_legend=True
                       )

# ax = axes[0, 2]
# _ = categorical_scatter(data=tmp,
#                         ax=ax,
#                         coord_base=coord_base,
#                         hue='leiden_cons',
#                         text_anno='leiden_cons', 
#                         s=ds,
#                         labelsize=8,
#                         max_points=None,
#                         palette='tab20',
#                         scatter_kws={'rasterized':True},
#                         # legend_kws={'ncol':1}, 
#                         # show_legend=True
#                        )

ax = axes[0, 2]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='ClusterTissue',
                        text_anno='ClusterTissue', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1}, 
                        # show_legend=True
                       )


ax = axes[1, 0]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue=np.log10(tmp['CisLongContact']+1),
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )

ax = axes[1, 1]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue='mCGFrac',
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )

ax = axes[1, 2]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue='Cis/Trans',
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )



In [22]:
cc = ConsensusClustering(model=None,
                         n_neighbors=n_neighbors,
                         metric=metric,
                         min_cluster_size=min_cluster_size,
                         leiden_repeats=leiden_repeats,
                         leiden_resolution=leiden_resolution,
                         consensus_rate=consensus_rate,
                         random_state=random_state,
                         train_frac=train_frac,
                         train_max_n=train_max_n,
                         max_iter=max_iter,
                         n_jobs=n_jobs,
                         target_accuracy=target_accuracy)


In [23]:
cc.fit_predict(adata.obsm[f'5kCG_u{npc_cg}_seuratL2'])

In [24]:
adata.obs['leiden_cons'] = cc.label.copy()
adata.obs['leiden_cv'] = cc.cv_predicted_label.copy()


In [25]:
if len(cc.step_data)>1:
    cc.target_accuracy = 0
    cc.supervise_learning()
    cc.final_evaluation()
    adata.obs['leiden_init'] = cc.label.copy()
else:
    adata.obs['leiden_init'] = adata.obs['leiden_cons'].copy()
    

In [26]:
cc.save('ConcensusClustering_mc.model.lib')
adata.write_h5ad('5kCG_embed.h5ad')


In [27]:
from sklearn.metrics import adjusted_rand_score as ARI
ARI(adata.obs['leiden_cons'], adata.obs['leiden_cv'])


In [28]:
ds = 2
coord_base = 'tsne'
adata.obsm[f'X_{coord_base}'] = adata.obsm[f'5kCG_u{npc_cg}_seuratL2_{coord_base}'].copy()
dump_embedding(adata, coord_base)

fig, axes = plt.subplots(2, 2, figsize=(10, 8), dpi=300, constrained_layout=True)

tmp = adata.obs.copy()
ax = axes[0,0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='leiden_cons',
                        text_anno='leiden_cons',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # show_legend=True
                       )

ax = axes[0,1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='leiden_cv',
                        text_anno='leiden_cv',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # show_legend=True
                       )

ax = axes[1,0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Tissue',
                        # text_anno='celltype', 
                        s=ds,
                        labelsize=6,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True
                       )

ax = axes[1,1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='ClusterTissue',
                        text_anno='ClusterTissue', 
                        s=ds,
                        labelsize=6,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1}, 
                        # show_legend=True
                       )

# plt.savefig(f'{group_name}_5kCG100k3C_cluster.pdf', transparent=True)


In [29]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score


In [30]:
data = normalize(adata.obsm[f'5kCG_u{npc_cg}_seuratL2'], axis=1)
clf = LogisticRegression(class_weight='balanced')
skf = StratifiedKFold(n_splits=3, shuffle=True)
label = adata.obs['leiden_init'].copy()
leg = np.sort(label.unique())
while len(leg)>1:
    count = label.astype(str).value_counts()
    score = np.zeros((2, len(leg), len(leg)))
    for i in np.arange(len(leg)-1):
        for j in np.arange(i+1, len(leg)):
            selc = label.isin([leg[i], leg[j]])
            if count.loc[leg[i]]<count.loc[leg[j]]:
                label_dict = {leg[i]:1, leg[j]:0}
            else:
                label_dict = {leg[i]:0, leg[j]:1}
            # selc = np.random.permutation(np.where(selc)[0])
            X = data[selc]
            y = label.values[selc].map(label_dict)
            pred = np.zeros(len(y))
            for train_index, test_index in skf.split(X, y):
                X_train, X_test, y_train, y_test = X[train_index], X[test_index], y[train_index], y[test_index]
                pred[test_index] = clf.fit(X_train, y_train).predict_proba(X_test)[:,1]
            score[0,i,j] = roc_auc_score(y, pred)
            score[1,i,j] = average_precision_score(y, pred)
    score = np.min(score, axis=0)
    min_score = np.min(score[np.triu_indices(len(leg), k=1)])
    if min_score>0.9:
        break
    else:
        xx,yy = np.where(score==min_score)
        xx, yy = leg[xx[0]], leg[yy[0]]
        print(f'Merge {xx} {yy}')
        label[label==xx] = yy
        leg = np.sort(label.unique())

count = label.astype(str).value_counts()
labelmap = count.reset_index().reset_index().set_index('leiden_init')['index'].to_dict()
label_reorder = label.map(labelmap).astype(int)
label_reorder = 'c' + label_reorder.astype(str)
np.save('mergemcg_rocpr.npy', label_reorder)


In [31]:
ds = 2
coord_base = 'tsne'
label_reorder = np.load('mergemcg_rocpr.npy', allow_pickle=True)

fig, axes = plt.subplots(3, 3, figsize=(15, 12), dpi=300, constrained_layout=True)

for i,xx in enumerate([adata_all, adata_mc, adata_3c]):
    dump_embedding(xx, coord_base)
    tmp = xx.obs.copy()
    tmp['leiden_init'] = label_reorder.copy()
    tmp['ClusterTissue'] = adata.obs['ClusterTissue'].copy()
    ax = axes[0, i]
    _ = categorical_scatter(data=tmp,
                            ax=ax,
                            coord_base=coord_base,
                            hue='Donor',
                            s=ds,
                            labelsize=8,
                            max_points=None,
                            # palette='tab20',
                            scatter_kws={'rasterized':True},
                            show_legend=True)

    ax = axes[1, i]
    _ = categorical_scatter(data=tmp,
                            ax=ax,
                            coord_base=coord_base,
                            hue='leiden_init',
                            text_anno='leiden_init', 
                            s=ds,
                            labelsize=8,
                            max_points=None,
                            palette='tab20',
                            scatter_kws={'rasterized':True},
                            legend_kws={'ncol':1}, 
                            show_legend=True)

    ax = axes[2, i]
    _ = categorical_scatter(data=tmp,
                            ax=ax,
                            coord_base=coord_base,
                            hue='ClusterTissue',
                            text_anno='ClusterTissue', 
                            s=ds,
                            labelsize=8,
                            max_points=None,
                            palette='tab20',
                            scatter_kws={'rasterized':True},
                            # legend_kws={'ncol':1}, 
                            # show_legend=False
                           )
